# Implementing tokenizer from scratch

## What is a tokenizer?
<div class="alert alert-block alert-success">
A component that breaks down raw text into smaller units called tokens, which are the basic building blocks that language models actually process.
</div>

For demonstration purpose, we work at small datasets. However, in the real world, it is common to preprocess a large amount of articles or books in order to train LLMs.

Firstly, let's read and inspect the text included here!

In [28]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


We are going to use Python regular expresseion library **re** for this implementation to help understand how tokenizer works.

Without further ado, let's go!

## Step 1: Creating tokens

In [29]:
import re
# Importing library

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


<div class="alert alert-block alert-success">
Let's use AI to break down everything happened here!
</div>
---

### 1. `import re`
- This imports Python’s **regular expression (regex)** library.
- Regex lets you define complex search/split patterns for strings.

---

### 2. `re.split(r'([.,:;?_"()\'`|--|\s)', raw_text)`
- **`re.split(pattern, string)`**: Splits `string` wherever the regex `pattern` matches.
- The pattern here is:
  ```
  r'([.,:;?_"()\'`|--|\s])'
  ```
  - `r''`: Raw string literal, so backslashes are treated literally.
  - `[ ... ]`: A **character class**, meaning “match any one of these characters.”
  - Inside the brackets:
    - `.,:;?_"()\'\`` → punctuation marks
    - `--` → includes double dashes
    - `\s` → matches any whitespace (spaces, tabs, newlines).
  - The **parentheses `( ... )`** around the character class make it a **capturing group**.  
    → This means the delimiters (punctuation/whitespace) are also returned in the split list, not discarded.

---

### 3. List comprehension
```python
[item.strip() for item in preprocessed if item.strip()]
```
- Iterates through each `item` in the list `preprocessed`.
- `item.strip()` removes leading/trailing whitespace.
- `if item.strip()` ensures only non-empty strings are kept.
- Result: a cleaned list of tokens (words and punctuation).

---

### 4. `print(preprocessed[:30])`
- `[:30]` is **list slicing**: takes the first 30 elements.
- Prints them for inspection.

---

### ⚡ Key Takeaways
- **Regex split with capturing group** → keeps punctuation as separate tokens.
- **List comprehension with `strip()`** → cleans and filters out empty strings.
- **Slice `[:30]`** → limits output for readability.

---
### Fun fact
The actual training of the large language model has a vocabulary list that contains sub-words, like:
`(tokenizer's amazing)` may consist of tokens like: `(["token", "izer", "'s", "amaz", "ing")`
For those new to code:
<div class="alert alert-block alert-warning">
Don't worry if the code is hard to understand. Everyone starts like a noob. This notebook is designed to fully understand LLM, feel free to ask any questions to AI if you have any doubts. What really matter is to understand the nuts and bolts of everything to eventually become confident.
</div>

## Step 2: Creating Token IDs
## What is Token ID?
<div class='alert alert-block alert-success'>
Token IDs are just basically an identification of tokens: the numeric index (integer) assigned to a particular token in the tokenizer's fixed vocabulary. It is the actual value that a language model receives and processes.
</div>

Let's see the vocablary size first!

In [30]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


Emmm, quite a few vocabulary in this dataset!

Now we will code a **vocabulary list**. It is a Python dictionary. Note that it has a result of `{token: integer}` as it is meant for **encoding** which will be covered later.

Since it is quite large, we are only going to print the first 25 entries in the list.

In [31]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [32]:
for i, item in enumerate(vocab.items()):#Loops through the dictionary entries (key, value pairs).
                                        #enumerate adds a counter i to track how many items have been seen.
    print(item)
    if i >= 24:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)


As we can see, the dictionary contains every unique integer labels that correspond to every individual tokens.

In LLM, the model will:
- Firstly **endode** the tokens into token IDs:
    
    splitting text into token, carrying out the string-to-integer mapping to produce token IDs into the dictionary which are going to be our vocabulary list.
- Then **decode** the tokens from their token IDs:
    
    Carrying out the integer-to-string mapping to produce token IDs back to text.

We are going to create a class for this, where we already have an intuition of what the class should include: **Encoder** and **Decoder**.
## 3. Coding a tokenizer class in Python

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        #Store the vocabulary as a class attribute for access in the encode and decode methods
        self.str_to_int = vocab # ENCODING
        #Note that vocab = {token:integer for integer, token in enumerate(all_words)}
        #Create an INVERSE vocabulary that maps token IDs back to the original text tokens
        self.int_to_str = {i:s for s, i in vocab.items()} # DECODING
    
    #Process input text into token IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        #Splits input text into tokens using regex (punctuation + whitespace as separators)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        #Cleans tokens with strip() and filters out empty strings.
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    #Convert token IDs back into tokens
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        #Joins them with spaces.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        #Uses re.sub to remove unwanted spaces before punctuation (so you don’t get "Hello !", but "Hello!").
        return text

Create an instance and try out!

In [34]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
#Notice this text snippet is directly copy pasted from the story
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


<div class='alert alert-block alert-success'>
Looks cool!

Try converting back to tokens:
</div>

In [35]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

<div class='alert alert-block alert-success'>
Not exactly the same but we can see that the decode method successfully converted the token IDs back into the original text.
</div>